# Melt Budget Data Loading
**Notebook 1 of 2** — run before `sankey_figures.ipynb`.

Loads, area-integrates, and saves CMIP6 and CESM2-LE ice and snow mass budget variables as `.nc` files to `save_path`. Processes one CMIP6 model at a time (set `models` below). The saved files are read directly by `sankey_figures.ipynb`.

In [1]:
# Install these packages once per session on the cryohub.
# All packages are used in the CMIP6 and CESM loading code: files/load.py
# Skip if these packages are already installed in your environment

# ── Core climate/geospatial dependencies ──────────────────────────────────────
%pip install -q regionmask       # regional masking for geospatial data
%pip install -q bottleneck       # speeds up xarray reduce operations (rolling mean, etc.)
%pip install -q xesmf            # regridding for Earth System Model output
%pip install -q cf_xarray        # CF convention accessor for xarray datasets

# ── ESGF data access ──────────────────────────────────────────────────────────
%pip install -q esgf-pyclient    # search client for the Earth System Grid Federation
%pip install -q intake-esgf      # intake driver for ESGF catalogs

# ── Version-pinned installs (workarounds) ─────────────────────────────────────
# importlib_metadata<8 fixes an import error with esmpy
# may no longer be needed    
#%pip install -q "importlib_metadata<8"

# globus-sdk must stay below v4 until intake-esgf adds compatibility
%pip install -q "globus-sdk<4"

# pydantic upgrade needed for intake-esm compatibility
%pip install -q --upgrade intake-esm pydantic

# numpy pinned to <=2.3 due to numba incompatibility
%pip install -q "numpy<=2.3"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmip 0.7.2 requires xarrayutils, which is not installed.
xmip 0.7.2 requires xgcm<0.7.0, which is not installed.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is 

**Restart the kernel after the installs above before running any further cells.**

In [1]:
import os
import sys
import logging
from io import StringIO
import numpy as np
import xarray as xr
import intake
from dask.diagnostics import ProgressBar
from load import CMIP6, CESM
from functions import region_mask,preferred_load_list,to_pystr_list
import time

In [2]:
# Directory where area-integrated .nc files are written (must match melt_path in sankey_figures.ipynb)
save_path = '/home/jovyan/shared-public/ICESat-2-sea-ice/better_output/melt/'
save = False  # set True to write files

cat_url  = 'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'
cat_url2 = 'https://storage.googleapis.com/cmip6/cmip6-pgf-ingestion-test/catalog/catalog.json'

args = {
    'verbose': True,
    'skip_sids': ['CAS-ESM2-0', 'GISS-E2-1-G-CC', 'GISS-E2-2-G', 'GISS-E2-1-G', 'GISS-E2-1-H'],  # models with known data issues
    'grid_label': ['gn', 'gr'],
    'members': 'all',
    'experiment_id': ['historical'],
    'time_chunks': 200,       # months per dask chunk
    'esgf_url': 'https://esgf-node.ornl.gov/esg-search',
    'cat_url': cat_url,
    'cat_url2': cat_url2,
}

In [16]:
variables = ['sidmassmelttop', 'sidmassmeltbot', 'sidmasslat', 'sidmassgrowthbot', 'sidmassgrowthwat', 'sidmasssi', 'sidmassevapsubl', 'sidmassdyn'
             , 'sndmassmelt' , 'sndmasssnf', 'sndmasssi', 'sndmasssubl', 'sndmasswindrif', 'sndmassdyn']

descriptions = [
    '(Sea-Ice Mass Change Through Surface Melting [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Bottom Melting [kg m-2 s-1])',
    '(Lateral Sea Ice Melt Rate [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Basal Growth [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Growth in Supercooled Open Water (Frazil) [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Snow-to-Ice Conversion [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Evaporation and Sublimation [kg m-2 s-1])',
    '(Sea-Ice Mass Change from Dynamics [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Melt [kg m-2 s-1])',
    '(Snow Mass Change Through Snow Fall [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Snow-to-Ice Conversion [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Evaporation or Sublimation [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Wind Drift of Snow [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Advection by Sea-Ice Dynamics [kg m-2 s-1])'
]

In [5]:
cat_cloud_gn_ssp245 = {}
cat_esgf_gn_ssp245 = {}

for var in variables:
    start = time.time()
    cat_cloud_gn_ssp245[var], cat_esgf_gn_ssp245[var] = CMIP6(grid_label=['gn'], members='all', experiment_id = ['ssp245']
                                          , variable=var, table_id='SImon').data_summary()
    elapsed = time.time() - start
    print(f"✅ {var} search completed in {elapsed:.2f} seconds\n")

✅ sidmassmelttop search completed in 62.97 seconds

✅ sidmassmeltbot search completed in 219.93 seconds

✅ sidmasslat search completed in 344.61 seconds

✅ sidmassgrowthbot search completed in 8.23 seconds

✅ sidmassgrowthwat search completed in 7.85 seconds

✅ sidmasssi search completed in 7.25 seconds

✅ sidmassevapsubl search completed in 7.62 seconds

✅ sidmassdyn search completed in 7.40 seconds

✅ sndmassmelt search completed in 6.59 seconds

✅ sndmasssnf search completed in 9.09 seconds

✅ sndmasssi search completed in 6.44 seconds

✅ sndmasssubl search completed in 6.07 seconds

✅ sndmasswindrif search completed in 5.85 seconds

✅ sndmassdyn search completed in 7.55 seconds



In [14]:
models_gn_ssp245 = {
    var: preferred_load_list(cat_cloud_gn_ssp245[var], cat_esgf_gn_ssp245[var])
    for var in variables
}

availability_count = {key: {subkey: len(value) for subkey, value in subdict.items()} for key, subdict in models_gn_ssp245.items()}
availability_count_df = pd.DataFrame(availability_count).transpose()
availability_count_df

,cloud,esgf
sidmassmelttop,0,29
sidmassmeltbot,0,29
sidmasslat,0,21
sidmassgrowthbot,0,27
sidmassgrowthwat,0,28
sidmasssi,0,29
sidmassevapsubl,0,27
sidmassdyn,0,21
sndmassmelt,0,24
sndmasssnf,17,15


In [17]:
availability_desc = {var+'  '+desc: models_gn_ssp245[var] for var,desc in zip(variables,descriptions)}

availability_df = pd.DataFrame(availability_desc).transpose().style.set_properties(**{
    'white-space': 'normal'
})
availability_df

,cloud,esgf
sidmassmelttop (Sea-Ice Mass Change Through Surface Melting [kg m-2 s-1]),[],"['ACCESS-CM2', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'TaiESM1', 'UKESM1-0-LL']"
sidmassmeltbot (Sea-Ice Mass Change Through Bottom Melting [kg m-2 s-1]),[],"['ACCESS-CM2', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'TaiESM1', 'UKESM1-0-LL']"
sidmasslat (Lateral Sea Ice Melt Rate [kg m-2 s-1]),[],"['ACCESS-CM2', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL']"
sidmassgrowthbot (Sea-Ice Mass Change Through Basal Growth [kg m-2 s-1]),[],"['ACCESS-CM2', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'TaiESM1', 'UKESM1-0-LL']"
sidmassgrowthwat (Sea-Ice Mass Change Through Growth in Supercooled Open Water (Frazil) [kg m-2 s-1]),[],"['ACCESS-CM2', 'BCC-CSM2-MR', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'TaiESM1', 'UKESM1-0-LL']"
sidmasssi (Sea-Ice Mass Change Through Snow-to-Ice Conversion [kg m-2 s-1]),[],"['ACCESS-CM2', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'FGOALS-f3-L', 'FGOALS-g3', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL']"
sidmassevapsubl (Sea-Ice Mass Change Through Evaporation and Sublimation [kg m-2 s-1]),[],"['ACCESS-CM2', 'ACCESS-ESM1-5', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-CC', 'EC-Earth3-Veg', 'EC-Earth3-Veg-LR', 'GISS-E2-1-G', 'GISS-E2-1-G-CC', 'GISS-E2-1-H', 'GISS-E2-2-G', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL']"
sidmassdyn (Sea-Ice Mass Change from Dynamics [kg m-2 s-1]),[],"['ACCESS-CM2', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'FGOALS-f3-L', 'FGOALS-g3', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0', 'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL']"
sndmassmelt (Snow Mass Rate of Change Through Melt [kg m-2 s-1]),[],"['ACCESS-CM2', 'BCC-CSM2-MR', 'CAS-ESM2-0', 'CESM2', 'CESM2-WACCM', 'CIESM', 'CMCC-CM2-SR5', 'CMCC-ESM2', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-E

In [10]:
# Redirect stderr to suppress unwanted warnings
sys.stderr = err = StringIO()
# Suppress specific Google and urllib3 loggers
for name in logging.Logger.manager.loggerDict.keys():
    if ('google' in name) or ('url' in name):
        logger = logging.getLogger(name)
        logger.setLevel(logging.CRITICAL)
        logger.propagate = False  # Prevent propagation to root logger
        for handler in logger.handlers:  # Remove existing handlers
            logger.removeHandler(handler)

## CMIP6

Load one model at a time. Available models: `ACCESS-CM2`, `HadGEM3-GC31-LL`, `UKESM1-0-LL`, `NorESM2-LM`, `NorESM2-MM`. Some models required corrections to standardize output to a common convention.

In [4]:
models = ['NorESM2-LM']  # currently only works when set to one model
if isinstance(models, str): models = [models]

### Load gridded budget variables

Due to the changes made to the ESGF archive and the lack of budget variables in the cloud, most of this data will be cached locally to .esgf_manual and must be deleted manually after use. Data may once again be streamable in the future after data migration stabilizes. 

In [ ]:
# CMIP6 variable naming: sidmass* = sea ice mass flux (kg m⁻² s⁻¹), sndmass* = snow mass flux
CMIP6_siconc    = CMIP6(**args, variable='siconc',           source_id=models, table_id='SImon').load_data()

CMIP6_thermo       = CMIP6(**args, variable='sidmassth',         source_id=models, table_id='SImon').load_data()
CMIP6_basal_growth = CMIP6(**args, variable='sidmassgrowthbot',  source_id=models, table_id='SImon').load_data()
CMIP6_frazil       = CMIP6(**args, variable='sidmassgrowthwat',  source_id=models, table_id='SImon').load_data()
CMIP6_snow_ice     = CMIP6(**args, variable='sidmasssi',         source_id=models, table_id='SImon').load_data()
CMIP6_top_melt     = CMIP6(**args, variable='sidmassmelttop',    source_id=models, table_id='SImon').load_data()
CMIP6_basal_melt   = CMIP6(**args, variable='sidmassmeltbot',    source_id=models, table_id='SImon').load_data()
CMIP6_lateral_melt = CMIP6(**args, variable='sidmasslat',        source_id=models, table_id='SImon').load_data()
CMIP6_evap_subl    = CMIP6(**args, variable='sidmassevapsubl',   source_id=models, table_id='SImon').load_data()
CMIP6_dynamics     = CMIP6(**args, variable='sidmassdyn',        source_id=models, table_id='SImon').load_data()

CMIP6_snowfall      = CMIP6(**args, variable='sndmasssnf',      source_id=models, table_id='SImon').load_data()
CMIP6_snowmelt      = CMIP6(**args, variable='sndmassmelt',     source_id=models, table_id='SImon').load_data()
CMIP6_snow_ice_snow = CMIP6(**args, variable='sndmasssi',       source_id=models, table_id='SImon').load_data()
CMIP6_snow_dynamics = CMIP6(**args, variable='sndmassdyn',      source_id=models, table_id='SImon').load_data()
CMIP6_wind_drift    = CMIP6(**args, variable='sndmasswindrif',  source_id=models, table_id='SImon').load_data()
CMIP6_evap_subl_snow = CMIP6(**args, variable='sndmasssubl',   source_id=models, table_id='SImon').load_data()

### Area-integrate snow fluxes (10³ Gt month⁻¹)

Model notes:

- ACCESS-CM2:
    - All variables are defined per sea ice area and were converted to grid cell area
    - It is not a documented error, but the snowmelt appears much larger than physically possible. In an attempt to fix, snowmelt terms were divided by 3.3 (snow density/100).
    - sndmasswindrif and sndmasssubl are unavailable
- HadGEM3-GC31-LL and UKESM1-0-LL
    - sndmasswindrif and sndmasssubl are unavailable
- NorESM-LM and MM
    - Snowmelt was defined positive and switched to a negative flux.
    - Snowfall also need to be divided by snow density (330).
    - sndmasssubl and sndmassdyn are unavailable

In [ ]:
seconds = CMIP6_siconc.time.dt.days_in_month * 86400

if models[0] == 'ACCESS-CM2':
    CMIP6_snowmelt_SH      = (((CMIP6_snowmelt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/3.3
    CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/3.3
    CMIP6_snowmelt_NH      = (((CMIP6_snowmelt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/3.3
    
    CMIP6_snowfall_SH      = (((CMIP6_snowfall*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    CMIP6_snowfall_NH      = (((CMIP6_snowfall*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
    CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
    CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
    CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
    CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn

if models[0] in ['HadGEM3-GC31-LL','UKESM1-0-LL']:
    CMIP6_snowmelt_SH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    CMIP6_snowmelt_NH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
    CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
    CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
    CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
    CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
    CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn

if models[0] in ['NorESM2-LM','NorESM2-MM']:

    CMIP6_snowmelt_SH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    CMIP6_snowmelt_Weddell = -((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    CMIP6_snowmelt_NH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
    CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330
    CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330
    CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330

    CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
    CMIP6_wind_drift_SH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
    CMIP6_wind_drift_Weddell = ((region_mask(CMIP6_wind_drift,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
    CMIP6_wind_drift_NH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif

#keep in case more models are loaded that have sndmasssubl
#CMIP6_evap_subl_snow_SH      = (((CMIP6_evap_subl_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
#CMIP6_evap_subl_snow_Weddell = ((region_mask(CMIP6_evap_subl_snow*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
#CMIP6_evap_subl_snow_NH      = (((CMIP6_evap_subl_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl

### Area-integrate ice fluxes (10³ Gt month⁻¹)

Model notes:

- ACCESS-CM2:
    - Thermodynamics and dynamics are defined relative to grid cell area
    - All other variables are incorrectly defined relative to sea ice area and need to be scaled by `siconc/100` before computing the spatial sum.
    - sidmassevapsubl (evaporation and sublimation) was defined positive in the model output and has been changed to negative
- HadGEM3-GC31-LL and UKESM1-0-LL
    - sidmassevapsubl (evaporation and sublimation) was defined positive in the model output and has been changed to negative
- NorESM-LM and MM
    - Melt terms (top melt: sidmassmelttop, basal melt: sidmassmeltbot, and lateral melt: sidmasslat) were defined positive in the model output and have been changed to negative

In [ ]:
if models[0] == 'ACCESS-CM2':

    CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
    CMIP6_basal_growth_SH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_NH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
    CMIP6_frazil_SH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_NH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
    CMIP6_snow_ice_SH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_NH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
    CMIP6_top_melt_SH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_NH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
    CMIP6_basal_melt_SH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_NH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
    CMIP6_lateral_melt_SH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_NH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_Weddell = ((region_mask(CMIP6_lateral_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
    CMIP6_evap_subl_SH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_NH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
    CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
if models[0] in ['HadGEM3-GC31-LL','UKESM1-0-LL']:
    
    CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
    CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
    CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
    CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
    CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
    CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
    CMIP6_lateral_melt_SH = (((CMIP6_lateral_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_NH = (((CMIP6_lateral_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_Weddell = ((region_mask(CMIP6_lateral_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
    CMIP6_evap_subl_SH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_NH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
    CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
if models[0] in ['NorESM2-LM','NorESM2-MM']:

    CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
    CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
    CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
    CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
    CMIP6_top_melt_SH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_NH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    CMIP6_top_melt_Weddell = -((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
    CMIP6_basal_melt_SH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_NH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    CMIP6_basal_melt_Weddell = -((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
    CMIP6_lateral_melt_SH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_NH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    CMIP6_lateral_melt_Weddell = -((region_mask(CMIP6_lateral_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
    CMIP6_evap_subl_SH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_NH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
    CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn

### Merge and save

In [ ]:
CMIP6_snowfall_ds = xr.merge([CMIP6_snowfall_SH.to_dataset(name='snowfall_SH'),
                        CMIP6_snowfall_NH.to_dataset(name='snowfall_NH'),
                        CMIP6_snowfall_Weddell.to_dataset(name='snowfall_Weddell')])
CMIP6_snowmelt_ds = xr.merge([CMIP6_snowmelt_SH.to_dataset(name='snowmelt_SH'),
                        CMIP6_snowmelt_NH.to_dataset(name='snowmelt_NH'),
                        CMIP6_snowmelt_Weddell.to_dataset(name='snowmelt_Weddell')])
CMIP6_snow_ice_snow_ds = xr.merge([CMIP6_snow_ice_snow_SH.to_dataset(name='snow_ice_snow_SH'),
                             CMIP6_snow_ice_snow_NH.to_dataset(name='snow_ice_snow_NH'),
                             CMIP6_snow_ice_snow_Weddell.to_dataset(name='snow_ice_snow_Weddell')])
if CMIP6_snow_dynamics is not None:
    CMIP6_snow_dynamics_ds = xr.merge([CMIP6_snow_dynamics_SH.to_dataset(name='snow_dynamics_SH'),
                                 CMIP6_snow_dynamics_NH.to_dataset(name='snow_dynamics_NH'),
                                 CMIP6_snow_dynamics_Weddell.to_dataset(name='snow_dynamics_Weddell')])
if CMIP6_wind_drift is not None:
    CMIP6_wind_drift_ds = xr.merge([CMIP6_wind_drift_SH.to_dataset(name='wind_drift_SH'),
                              CMIP6_wind_drift_NH.to_dataset(name='wind_drift_NH'),
                              CMIP6_wind_drift_Weddell.to_dataset(name='wind_drift_Weddell')])
if CMIP6_evap_subl_snow is not None:
    CMIP6_evap_subl_snow_ds = xr.merge([CMIP6_evap_subl_snow_SH.to_dataset(name='evap_subl_snow_SH'),
                                  CMIP6_evap_subl_snow_NH.to_dataset(name='evap_subl_snow_NH'),
                                  CMIP6_evap_subl_snow_Weddell.to_dataset(name='evap_subl_snow_Weddell')])

CMIP6_thermo_ds = xr.merge([CMIP6_thermo_SH.to_dataset(name='thermo_SH')
          ,CMIP6_thermo_NH.to_dataset(name='thermo_NH')
          ,CMIP6_thermo_Weddell.to_dataset(name='thermo_Weddell')])
CMIP6_basal_growth_ds = xr.merge([CMIP6_basal_growth_SH.to_dataset(name='basal_growth_SH')
          ,CMIP6_basal_growth_NH.to_dataset(name='basal_growth_NH')
          ,CMIP6_basal_growth_Weddell.to_dataset(name='basal_growth_Weddell')])
CMIP6_frazil_ds = xr.merge([CMIP6_frazil_SH.to_dataset(name='frazil_SH')
          ,CMIP6_frazil_NH.to_dataset(name='frazil_NH')
          ,CMIP6_frazil_Weddell.to_dataset(name='frazil_Weddell')])
CMIP6_snow_ice_ds = xr.merge([CMIP6_snow_ice_SH.to_dataset(name='snow_ice_SH')
          ,CMIP6_snow_ice_NH.to_dataset(name='snow_ice_NH')
          ,CMIP6_snow_ice_Weddell.to_dataset(name='snow_ice_Weddell')])
CMIP6_top_melt_ds = xr.merge([CMIP6_top_melt_SH.to_dataset(name='top_melt_SH')
          ,CMIP6_top_melt_NH.to_dataset(name='top_melt_NH')
          ,CMIP6_top_melt_Weddell.to_dataset(name='top_melt_Weddell')])
CMIP6_basal_melt_ds = xr.merge([CMIP6_basal_melt_SH.to_dataset(name='basal_melt_SH')
          ,CMIP6_basal_melt_NH.to_dataset(name='basal_melt_NH')
          ,CMIP6_basal_melt_Weddell.to_dataset(name='basal_melt_Weddell')])
CMIP6_lateral_melt_ds = xr.merge([CMIP6_lateral_melt_SH.to_dataset(name='lateral_melt_SH')
          ,CMIP6_lateral_melt_NH.to_dataset(name='lateral_melt_NH')
          ,CMIP6_lateral_melt_Weddell.to_dataset(name='lateral_melt_Weddell')])
CMIP6_evap_subl_ds = xr.merge([CMIP6_evap_subl_SH.to_dataset(name='evap_subl_SH')
          ,CMIP6_evap_subl_NH.to_dataset(name='evap_subl_NH')
          ,CMIP6_evap_subl_Weddell.to_dataset(name='evap_subl_Weddell')])
CMIP6_dynamics_ds = xr.merge([CMIP6_dynamics_SH.to_dataset(name='dynamics_SH')
          ,CMIP6_dynamics_NH.to_dataset(name='dynamics_NH')
          ,CMIP6_dynamics_Weddell.to_dataset(name='dynamics_Weddell')])

In [ ]:
if save==True:
    CMIP6_snowfall_ds.to_netcdf(save_path + models[0] + '_snowfall_2015_2100.nc')
    CMIP6_snowmelt_ds.to_netcdf(save_path + models[0] + '_snowmelt_2015_2100.nc')
    CMIP6_snow_ice_snow_ds.to_netcdf(save_path + models[0] + '_snow_ice_snow_2015_2100.nc')
    if CMIP6_snow_dynamics is not None:
        CMIP6_snow_dynamics_ds.to_netcdf(save_path + models[0] + '_snow_dynamics_2015_2100.nc')
    if CMIP6_wind_drift is not None:
        CMIP6_wind_drift_ds.to_netcdf(save_path + models[0] + '_wind_drift_2015_2100.nc')
    if CMIP6_evap_subl_snow is not None:
        CMIP6_evap_subl_snow_ds.to_netcdf(save_path + models[0] + '_evap_subl_snow_2015_2100.nc')

    CMIP6_thermo_ds.to_netcdf(save_path+models[0]+'_thermo_2015_2100.nc')
    CMIP6_basal_growth_ds.to_netcdf(save_path+models[0]+'_basal_growth_2015_2100.nc')
    CMIP6_frazil_ds.to_netcdf(save_path+models[0]+'_frazil_2015_2100.nc')
    CMIP6_snow_ice_ds.to_netcdf(save_path+models[0]+'_snow_ice_2015_2100.nc')
    CMIP6_top_melt_ds.to_netcdf(save_path+models[0]+'_top_melt_2015_2100.nc')
    CMIP6_basal_melt_ds.to_netcdf(save_path+models[0]+'_basal_melt_2015_2100.nc')
    CMIP6_lateral_melt_ds.to_netcdf(save_path+models[0]+'_lateral_melt_2015_2100.nc')
    CMIP6_evap_subl_ds.to_netcdf(save_path+models[0]+'_evap_subl_2015_2100.nc')
    CMIP6_dynamics_ds.to_netcdf(save_path+models[0]+'_dynamics_2015_2100.nc')

## CESM2-LE

Unlike CMIP6 models (loaded via ESGF/Pangeo), CESM2-LE data is accessed through the `CESM` class, which reads from local CryoCloud storage or THREDDS OPeNDAP. No catalog search is involved: set `local_path` to the directory containing the CESM2-LE time-series files, or enable `load_opendap` to stream remotely.

### Load gridded budget variables

In [3]:
load_opendap = False
load_locally = True
local_path = '/home/jovyan/shared-public/ICESat-2-sea-ice/better_output/LE2/b.e21'

In [ ]:
if load_locally==True:
    LE2_thermo = CESM(variable='sidmassth',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_dynamics = CESM(variable='sidmassdyn',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_basal_growth = CESM(variable='sidmassgrowthbot',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_frazil = CESM(variable='sidmassgrowthwat',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_snow_ice = CESM(variable='sidmasssi',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_top_melt = CESM(variable='sidmassmelttop',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_basal_melt = CESM(variable='sidmassmeltbot',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_lateral_melt = CESM(variable='sidmasslat',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_evap_subl = CESM(variable='sidmassevapsubl',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    
    # sea ice concentration
    LE2_SIC = CESM(variable='aice',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    
    LE2_snowmelt = CESM(variable='sndmassmelt',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_snowfall = CESM(variable='sndmasssnf',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
    LE2_evap_subl_snow = CESM(variable='sndmassubl',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()

In [4]:
if load_opendap==True:
    LE2_thermo = CESM(variable='sidmassth',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_dynamics = CESM(variable='sidmassdyn',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_basal_growth = CESM(variable='sidmassgrowthbot',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_frazil = CESM(variable='sidmassgrowthwat',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_snow_ice = CESM(variable='sidmasssi',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_top_melt = CESM(variable='sidmassmelttop',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_basal_melt = CESM(variable='sidmassmeltbot',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_lateral_melt = CESM(variable='sidmasslat',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_evap_subl = CESM(variable='sidmassevapsubl',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    # sea ice concentration
    LE2_SIC = CESM(variable='aice',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    
    LE2_snowmelt = CESM(variable='sndmassmelt',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_snowfall = CESM(variable='sndmasssnf',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
    LE2_evap_subl_snow = CESM(variable='sndmassubl',source_id='CESM2-LE',time_chunks=50
                             ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()

OPeNDAP sidmassth:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sidmassdyn:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sidmassgrowthbot:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sidmassgrowthwat:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sidmasssi:   0%|          | 0/100 [00:00<?, ?file/s]

...

OPeNDAP sidmassevapsubl:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP aice:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sndmassmelt:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sndmasssnf:   0%|          | 0/100 [00:00<?, ?file/s]

OPeNDAP sndmassubl:   0%|          | 0/100 [00:00<?, ?file/s]

### Preprocess

In [5]:
# Convert ice→snow mass transfer to snow-side sign convention
# conversion factor needed to convert ice mass to snow mass rate change
LE2_snow_ice_snow = (-LE2_snow_ice.sidmasssi * (330/917)).to_dataset(name='sndmasssi')

# snowmelt is stored as positive loss in CESM; negate for sign consistency
LE2_snowmelt = -LE2_snowmelt

### Area-integrate snow fluxes (10³ Gt month⁻¹)

In [ ]:
seconds = LE2_SIC.time.dt.days_in_month * 86400

LE2_snowmelt_SH = (((LE2_snowmelt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassmelt
LE2_snowmelt_Weddell = ((region_mask(LE2_snowmelt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassmelt
    
LE2_snowfall_SH = (((LE2_snowfall*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssnf
LE2_snowfall_Weddell = ((region_mask(LE2_snowfall*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssnf
    
LE2_snow_ice_snow_SH = (((LE2_snow_ice_snow*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssi
LE2_snow_ice_snow_Weddell = ((region_mask(LE2_snow_ice_snow*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssi
    
LE2_evap_subl_snow_SH = (((LE2_evap_subl_snow*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassubl
LE2_evap_subl_snow_Weddell = ((region_mask(LE2_evap_subl_snow*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassubl
    
LE2_snowmelt_NH = (((LE2_snowmelt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassmelt
LE2_snowfall_NH = (((LE2_snowfall*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssnf
LE2_snow_ice_snow_NH = (((LE2_snow_ice_snow*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssi
LE2_evap_subl_snow_NH = (((LE2_evap_subl_snow*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassubl

### Save snow budget

In [ ]:
if save==True:
    xr.merge([LE2_snowmelt_SH.to_dataset(name='snowmelt_SH')
              ,LE2_snowmelt_Weddell.to_dataset(name='snowmelt_Weddell'),
             LE2_snowmelt_NH.to_dataset(name='snowmelt_NH')]).to_netcdf(save_path+'LE2'+'_snowmelt_2015_2034.nc')
    
    xr.merge([LE2_snowfall_SH.to_dataset(name='snowfall_SH')
              ,LE2_snowfall_Weddell.to_dataset(name='snowfall_Weddell'),
             LE2_snowfall_NH.to_dataset(name='snowfall_NH')]).to_netcdf(save_path+'LE2'+'_snowfall_2015_2034.nc')
    
    xr.merge([LE2_snow_ice_snow_SH.to_dataset(name='snow_ice_snow_SH')
              ,LE2_snow_ice_snow_Weddell.to_dataset(name='snow_ice_snow_Weddell'),
             LE2_snow_ice_snow_NH.to_dataset(name='snow_ice_snow_NH')]).to_netcdf(save_path+'LE2'+'_snow_ice_snow_2015_2034.nc')
    xr.merge([LE2_evap_subl_snow_SH.to_dataset(name='evap_subl_snow_SH')
              ,LE2_evap_subl_snow_Weddell.to_dataset(name='evap_subl_snow_Weddell'),
             LE2_evap_subl_snow_NH.to_dataset(name='evap_subl_snow_NH')]).to_netcdf(save_path+'LE2'+'_evap_subl_snow_2015_2034.nc')

### Area-integrate ice fluxes (10³ Gt month⁻¹)

In [ ]:
LE2_thermo_SH = (((LE2_thermo*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassth
LE2_thermo_Weddell = ((region_mask(LE2_thermo*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassth

LE2_dynamics_SH = (((LE2_dynamics*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassdyn
LE2_dynamics_Weddell = ((region_mask(LE2_dynamics*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassdyn

LE2_basal_growth_SH = (((LE2_basal_growth*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot
LE2_basal_growth_Weddell = ((region_mask(LE2_basal_growth*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot

LE2_frazil_SH = (((LE2_frazil*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat
LE2_frazil_Weddell = ((region_mask(LE2_frazil*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat

LE2_snow_ice_SH = (((LE2_snow_ice*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasssi
LE2_snow_ice_Weddell = ((region_mask(LE2_snow_ice*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasssi

LE2_top_melt_SH = -(((LE2_top_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmelttop
LE2_top_melt_Weddell = -((region_mask(LE2_top_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmelttop

LE2_basal_melt_SH = -(((LE2_basal_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmeltbot
LE2_basal_melt_Weddell = -((region_mask(LE2_basal_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmeltbot

LE2_lateral_melt_SH = -(((LE2_lateral_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasslat
LE2_lateral_melt_Weddell = -((region_mask(LE2_lateral_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasslat

LE2_evap_subl_SH = (((LE2_evap_subl*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassevapsubl
LE2_evap_subl_Weddell = ((region_mask(LE2_evap_subl*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassevapsubl



LE2_thermo_NH = (((LE2_thermo*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassth
LE2_dynamics_NH = (((LE2_dynamics*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassdyn
LE2_basal_growth_NH = (((LE2_basal_growth*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot
LE2_frazil_NH = (((LE2_frazil*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat
LE2_snow_ice_NH = (((LE2_snow_ice*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasssi
LE2_top_melt_NH = -(((LE2_top_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmelttop
LE2_basal_melt_NH = -(((LE2_basal_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmeltbot
LE2_lateral_melt_NH = -(((LE2_lateral_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasslat
LE2_evap_subl_NH = (((LE2_evap_subl*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassevapsubl

### Save ice budget

In [ ]:
if save==True:
    xr.merge([LE2_thermo_SH.to_dataset(name='thermo_SH')
              ,LE2_thermo_Weddell.to_dataset(name='thermo_Weddell'),
             LE2_thermo_NH.to_dataset(name='thermo_NH')]).to_netcdf(save_path+'LE2'+'_thermo_2015_2034.nc')
    
    xr.merge([LE2_dynamics_SH.to_dataset(name='dynamics_SH')
              ,LE2_dynamics_Weddell.to_dataset(name='dynamics_Weddell'),
             LE2_dynamics_NH.to_dataset(name='dynamics_NH')]).to_netcdf(save_path+'LE2'+'_dynamics_2015_2034.nc')
    
    xr.merge([LE2_basal_growth_SH.to_dataset(name='basal_growth_SH')
              ,LE2_basal_growth_Weddell.to_dataset(name='basal_growth_Weddell'),
             LE2_basal_growth_NH.to_dataset(name='basal_growth_NH')]).to_netcdf(save_path+'LE2'+'_basal_growth_2015_2034.nc')
    
    xr.merge([LE2_frazil_SH.to_dataset(name='frazil_SH')
              ,LE2_frazil_Weddell.to_dataset(name='frazil_Weddell'),
             LE2_frazil_NH.to_dataset(name='frazil_NH')]).to_netcdf(save_path+'LE2'+'_frazil_2015_2034.nc')
    
    xr.merge([LE2_snow_ice_SH.to_dataset(name='snow_ice_SH')
              ,LE2_snow_ice_Weddell.to_dataset(name='snow_ice_Weddell'),
             LE2_snow_ice_NH.to_dataset(name='snow_ice_NH')]).to_netcdf(save_path+'LE2'+'_snow_ice_2015_2034.nc')
    
    xr.merge([LE2_top_melt_SH.to_dataset(name='top_melt_SH')
              ,LE2_top_melt_Weddell.to_dataset(name='top_melt_Weddell'),
             LE2_top_melt_NH.to_dataset(name='top_melt_NH')]).to_netcdf(save_path+'LE2'+'_top_melt_2015_2034.nc')
    
    xr.merge([LE2_basal_melt_SH.to_dataset(name='basal_melt_SH')
              ,LE2_basal_melt_Weddell.to_dataset(name='basal_melt_Weddell'),
             LE2_basal_melt_NH.to_dataset(name='basal_melt_NH')]).to_netcdf(save_path+'LE2'+'_basal_melt_2015_2034.nc')
    
    xr.merge([LE2_lateral_melt_SH.to_dataset(name='lateral_melt_SH')
              ,LE2_lateral_melt_Weddell.to_dataset(name='lateral_melt_Weddell'),
             LE2_lateral_melt_NH.to_dataset(name='lateral_melt_NH')]).to_netcdf(save_path+'LE2'+'_lateral_melt_2015_2034.nc')
    
    xr.merge([LE2_evap_subl_SH.to_dataset(name='evap_subl_SH')
              ,LE2_evap_subl_Weddell.to_dataset(name='evap_subl_Weddell'),
             LE2_evap_subl_NH.to_dataset(name='evap_subl_NH')]).to_netcdf(save_path+'LE2'+'_evap_subl_2015_2034.nc')